# GraphRAG — Knowledge Graph Retrieval

**Problem:** Some queries require multi-hop reasoning across entities.
- "What happens if Taiwan Semiconductor shuts down?"
- This needs: TSMC → supplies chips to → Apple, NVIDIA → their products → impact

Text search finds documents MENTIONING TSMC. Graph search TRAVERSES relationships
FROM TSMC to connected entities.

**Concept:** Build a knowledge graph where entities are nodes and relationships
are edges. For multi-hop queries, walk the graph to find connected information.

In [ ]:
# Build a simple knowledge graph from our supply chain data
# In production, this would be Neo4j with Cypher queries

graph = {
    "TSMC": {
        "type": "supplier",
        "country": "Taiwan",
        "supplies_to": ["Apple", "NVIDIA", "Qualcomm"],
        "produces": ["semiconductor chips"],
        "reliability": 94.7,
    },
    "Apple": {
        "type": "client",
        "depends_on": ["TSMC"],
        "products": ["iPhone", "MacBook", "iPad"],
        "disruption_impact": "production halts within 4 weeks",
    },
    "NVIDIA": {
        "type": "client",
        "depends_on": ["TSMC"],
        "products": ["GPU", "AI accelerators"],
        "disruption_impact": "supply drops by 60%",
    },
    "Shanghai Electronics": {
        "type": "supplier",
        "country": "China",
        "supplies_to": ["Automotive OEMs"],
        "produces": ["capacitors", "resistors"],
        "reliability": 87.3,
    },
    "Bosch": {
        "type": "supplier",
        "country": "Germany",
        "supplies_to": ["BMW", "Mercedes-Benz", "Toyota"],
        "produces": ["automotive sensors", "control units"],
        "reliability": 96.1,
    },
}

print("Knowledge Graph nodes:")
for entity, props in graph.items():
    print(f"  [{entity}] ({props['type']})")
    for key, val in props.items():
        if key != 'type':
            print(f"    → {key}: {val}")

In [ ]:
# Graph traversal: What happens if TSMC shuts down?
def traverse_impact(graph, entity, depth=0, visited=None):
    if visited is None:
        visited = set()
    if entity in visited or depth > 2:
        return []
    visited.add(entity)
    
    results = []
    indent = "  " * depth
    
    node = graph.get(entity, {})
    if node:
        results.append(f"{indent}[{entity}] ({node.get('type', 'unknown')})")
        
        # Follow supply chain connections
        for client in node.get("supplies_to", []):
            if client in graph:
                client_node = graph[client]
                impact = client_node.get("disruption_impact", "unknown impact")
                results.append(f"{indent}  → supplies to [{client}]: {impact}")
                results.extend(traverse_impact(graph, client, depth + 1, visited))
    
    return results

print("Query: 'What happens if TSMC shuts down?'\n")
print("Graph traversal (multi-hop):\n")
for line in traverse_impact(graph, "TSMC"):
    print(line)

In [ ]:
# Compare: Text search vs Graph search
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

docs = [
    "TSMC is the world's largest semiconductor foundry. Reliability: 94.7%.",
    "If TSMC shuts down, Apple iPhone production halts within 4 weeks.",
    "NVIDIA GPU supply drops by 60% if TSMC is disrupted.",
    "Bosch supplies automotive sensors to BMW and Mercedes.",
    "Shanghai Electronics has reliability 87.3% and supplies capacitors.",
]

query = "What happens if Taiwan Semiconductor shuts down?"
query_emb = model.encode(query)
doc_embs = model.encode(docs)

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Query: \"{query}\"\n")
print("Text search only finds docs mentioning TSMC directly:")
sims = [(i, cosine_sim(query_emb, d)) for i, d in enumerate(doc_embs)]
sims.sort(key=lambda x: x[1], reverse=True)
for idx, sim in sims:
    print(f"  {sim:.4f}: {docs[idx]}")

print("\nGraph search ALSO finds downstream impacts (Apple, NVIDIA)")
print("because it traverses: TSMC → supplies_to → Apple/NVIDIA → disruption_impact")

## In Production: Neo4j + Cypher

The in-memory graph above is a demo. In production, you'd use Neo4j:

```cypher
// Find all downstream impacts of a TSMC disruption
MATCH (tsmc:Supplier {name: 'TSMC'})-[:SUPPLIES_TO]->(client)
RETURN client.name, client.disruption_impact
```

## Key Takeaways

1. **Text search finds mentions** — GraphRAG finds **connections**
2. **Multi-hop queries need graph traversal** — "if X shuts down, what happens to Y?"
3. **Graph + Semantic search** = semantic finds seed docs, graph expands to connected entities
4. **Building the graph is the hard part** — entity extraction, relationship mapping
5. **Neo4j is the standard** for production knowledge graphs in RAG systems